# 0. NOTEBOOK HEADER & METADATA

## Notebook Metadata

| Field | Value |
|---|---|
| **Target Table** | `brk.dim_Booking` |
| **Product Group and Division** | To Be Updated By DJJ team |
| **Author** | Nayan Goyal |
| **Created** | 2026-04-03 |
| **DevOps ID and Link** | To Be Updated By DJJ team |
| **Load Type** | `Full Reload` |
| **Grain** | To Be Updated By DJJ team |

---

## Description

> To Be Updated By DJJ team

## Change Log

| Date | Author | DevOps ID and Link | Change Description |
|---|---|---|---|
|  |  |  |  |

# 1. CONFIGURATION, IMPORTS, & PARAMETERS

In [0]:
# ========================================
# LIBRARY IMPORTS
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from concurrent.futures import ThreadPoolExecutor, as_completed
from delta.tables import DeltaTable
from pyspark import StorageLevel
from datetime import datetime, timedelta
import uuid

In [0]:
# ==============================================
# CONFIGURATION
# ==============================================

SourceDatabaseName="djj_ods"                                    #Catalog name for source table
SourceSchemaName="dbo"                                          #Schema name for source table
SourceTableName="ODS_BRK_Booking"                               #Table name for source table
DestinationDatabaseName="djj_enriched_non_prod"                 #Catalog name for target table
DestinationSchemaName="brk"                                     #Schema name for target table
DestinationTableName="dim_booking"                                  #Table name for target table
etl_runtime = datetime.now()                                    #Current Runtime for updating LastUpdateTime

layer="enriched"                                                                            #Target layer
source_table=f"{SourceDatabaseName}.{SourceSchemaName}.{SourceTableName}"
target_table=f"{DestinationDatabaseName}.{DestinationSchemaName}.{DestinationTableName}"
proc="full_reload"                                                                    #Load type
platform="mssql"                                                                            #Source platform

federated_ods_catalog     = "djj_ods"
enriched_catalog    = "djj_enriched_non_prod"
federated_starschema_catalog  = "djj_starschema"
bronze_catalog = "bronze_non_prod"

# 2. START METADATA LOGGING

In [0]:
%run ../Metadata_Logger/execution_logger

In [0]:
# ===================================
# START METADATA LOGGING
# ===================================

run_ctx = start_execution_run(
        layer=layer,
        source_table=source_table,
        target_table=target_table,
        proc=proc,
        platform=platform
)

# 3. ETL LOGIC

In [0]:
# ============================================================
# SOURCE QUERY (BRK)
# ============================================================

ole_src_brk_booking_df = spark.sql(f"""
    SELECT BookingId, BookingNumber, VesselName, Voyage, TotalContainers, StatusCode, OBLNum, OBLDate, SailDate, ETADate, FundsReceiptDate 
    FROM {federated_ods_catalog}.dbo.ODS_BRK_Booking
""")

In [0]:
# ============================================================
# DERIVED COLUMNS (BRK)
# ============================================================

der_ods_brk_booking_output_df = ole_src_brk_booking_df\
    .withColumn("d_SalesOrderNo", F.lit("Unknown").cast(StringType()))\
    .withColumn("d_PONo", F.lit("Unknown").cast(StringType()))

In [0]:
# ============================================================
# LOOKUP TRANSFORMATION WITH ODS TABLE (BRK STATUS)
# ============================================================


def get_lkp_ods_brk_status_ref_df():
    df = spark.sql(f"""
        SELECT StatusCode, StatusDesc 
        FROM {federated_ods_catalog}.dbo.ODS_BRK_Status 
        WHERE BizType = 'B' 
        AND DocumentTypeCode = 'BKG'
    """)
    return df

lkp_ods_brk_status_ref_df = get_lkp_ods_brk_status_ref_df()

lkp_ods_brk_status_joined_df = der_ods_brk_booking_output_df.join(
    lkp_ods_brk_status_ref_df,
    der_ods_brk_booking_output_df["StatusCode"] == lkp_ods_brk_status_ref_df["StatusCode"],
    "left"
).select(
    der_ods_brk_booking_output_df["*"],
    lkp_ods_brk_status_ref_df["StatusDesc"]
)

In [0]:
# ============================================================
# SOURCE QUERY (MGEX)
# ============================================================

ole_src_mgex_mgbooking_df = spark.sql(f"""
    SELECT SalesOrderNo, PONo, BookingNo, VesselID, Voyage, Quantity, BookingStatus, EstDepartDate, EstArrivalDate 
    FROM {federated_ods_catalog}.dbo.ODS_MGEX_mgBooking
""")

In [0]:
# ============================================================
# DERIVED COLUMNS (MGEX)
# ============================================================

der_ods_mgex_mgbooking_output_df = ole_src_mgex_mgbooking_df\
    .withColumn("d_BookingID", F.lit(-1).cast(IntegerType()))\
    .withColumn("d_OBLNum", F.lit("Unknown").cast(StringType()))\
    .withColumn("d_OBLDate", F.lit("1900-01-01").cast(TimestampType()))\
    .withColumn("d_FundsReceiptDate", F.lit("1900-01-01").cast(TimestampType()))

In [0]:
# ============================================================
# LOOKUP TRANSFORMATION WITH ODS TABLE (MGEX VESSEL)
# ============================================================

def get_lkp_ods_mgex_mgvessel_ref_df():
    df = spark.sql(f"""
        SELECT VesselID, VesselName 
        FROM {federated_ods_catalog}.dbo.ODS_MGEX_mgVessel
    """)
    return df

lkp_ods_mgex_mgvessel_ref_df = get_lkp_ods_mgex_mgvessel_ref_df()

lkp_ods_mgex_mgvessel_joined_df = der_ods_mgex_mgbooking_output_df.join(
    lkp_ods_mgex_mgvessel_ref_df,
    der_ods_mgex_mgbooking_output_df["VesselID"] == lkp_ods_mgex_mgvessel_ref_df["VesselID"],
    "left"
).select(
    der_ods_mgex_mgbooking_output_df["*"],
    lkp_ods_mgex_mgvessel_ref_df["VesselName"]
)

In [0]:
# ============================================================
# LOOKUP TRANSFORMATION WITH ODS TABLE (MGEX BOOKING STATUS)
# ============================================================

def get_lkp_ods_mgex_mgbookingstatus_ref_df():
    df = spark.sql(f"""
        SELECT StatusID, Status 
        FROM {federated_ods_catalog}.dbo.ODS_MGEX_mgBookingStatus
    """)
    return df

lkp_ods_mgex_mgbookingstatus_ref_df = get_lkp_ods_mgex_mgbookingstatus_ref_df()

lkp_ods_mgex_mgbookingstatus_joined_df = lkp_ods_mgex_mgvessel_joined_df.join(
    lkp_ods_mgex_mgbookingstatus_ref_df,
    lkp_ods_mgex_mgvessel_joined_df["BookingStatus"] == lkp_ods_mgex_mgbookingstatus_ref_df["StatusID"],
    "left"
).select(
    lkp_ods_mgex_mgvessel_joined_df["*"],
    lkp_ods_mgex_mgbookingstatus_ref_df["Status"]
)

In [0]:
# ============================================================
# UNION ALL BY COLUMN NAME
# ============================================================

brk_branch_df = lkp_ods_brk_status_joined_df.select(
    F.col("BookingId").alias("BookingID"),
    F.col("BookingNumber"),
    F.col("VesselName"),
    F.col("Voyage"),
    F.col("TotalContainers"),
    F.col("OBLNum"),
    F.col("OBLDate"),
    F.col("SailDate").alias("EstDepartureDate"),
    F.col("ETADate").alias("EstArrivalDate"),
    F.col("StatusDesc").alias("Status"),
    F.col("d_PONo").alias("PONo"),
    F.col("d_SalesOrderNo").alias("SalesOrderNo"),
    F.col("FundsReceiptDate")
)

mgex_branch_df = lkp_ods_mgex_mgbookingstatus_joined_df.select(
    F.col("d_BookingID").alias("BookingID"),
    F.col("BookingNo").alias("BookingNumber"),
    F.col("VesselName"),
    F.col("Voyage"),
    F.col("Quantity").alias("TotalContainers"),
    F.col("d_OBLNum").alias("OBLNum"),
    F.col("d_OBLDate").alias("OBLDate"),
    F.col("EstDepartDate").alias("EstDepartureDate"),
    F.col("EstArrivalDate"),
    F.col("Status"),
    F.col("PONo"),
    F.col("SalesOrderNo"),
    F.col("d_FundsReceiptDate").alias("FundsReceiptDate")
)

ua_brk_dimbooking_output_df = mgex_branch_df.unionByName(brk_branch_df)

In [0]:
# ============================================================
# TRIM TRANSFORMATION ON COLUMNS
# ============================================================

tf_tp_brk_dimbooking_output_df = ua_brk_dimbooking_output_df\
    .withColumn("BookingNumber", F.trim(F.col("BookingNumber")))\
    .withColumn("VesselName", F.trim(F.col("VesselName")))\
    .withColumn("Voyage", F.regexp_replace(F.col("Voyage"), r'^[\s"]+|[\s"]+$', ''))\
    .withColumn("OBLNum", F.trim(F.col("OBLNum")))\
    .withColumn("Status", F.trim(F.col("Status")))

In [0]:
# ============================================================
# NULL HANDLING OF COLUMNS
# ============================================================

tf_nh_brk_dimbooking_output_df = tf_tp_brk_dimbooking_output_df.select(
    "BookingID",
    "BookingNumber",
    "VesselName",
    "Voyage",
    "TotalContainers",
    "OBLNum",
    "OBLDate",
    "EstDepartureDate",
    "EstArrivalDate",
    "Status",
    "SalesOrderNo",
    "PONo",
    "FundsReceiptDate"
)\
    .withColumn("BookingNumber", F.coalesce(F.col("BookingNumber"), F.lit("Unknown").cast(StringType())))\
    .withColumn("VesselName", F.coalesce(F.col("VesselName"), F.lit("Unknown").cast(StringType())))\
    .withColumn("Voyage", F.coalesce(F.col("Voyage"), F.lit("Unknown").cast(StringType())))\
    .withColumn("TotalContainers", F.coalesce(F.col("TotalContainers"), F.lit(0).cast(IntegerType())))\
    .withColumn("OBLNum", F.coalesce(F.col("OBLNum"), F.lit("Unknown").cast(StringType())))\
    .withColumn("OBLDate", F.coalesce(F.col("OBLDate"), F.lit("1900-01-01").cast(TimestampType())))\
    .withColumn("EstDepartureDate", F.coalesce(F.col("EstDepartureDate"), F.lit("1900-01-01").cast(TimestampType())))\
    .withColumn("EstArrivalDate", F.coalesce(F.col("EstArrivalDate"), F.lit("1900-01-01").cast(TimestampType())))\
    .withColumn("Status", F.coalesce(F.col("Status"), F.lit("Unknown").cast(StringType())))\
    .withColumn("FundsReceiptDate", F.coalesce(F.col("FundsReceiptDate"), F.lit("1900-01-01").cast(TimestampType())))

In [0]:
# ============================================================
# DERIVED METADATA COLUMNS
# ============================================================

variable_etl_run_time = F.current_timestamp()
der_metadata_output_df = tf_nh_brk_dimbooking_output_df\
    .withColumn("ETLSSExecutionID", F.lit(None).cast(IntegerType()))\
    .withColumn("DJJLastUpdateTime", F.lit(variable_etl_run_time).cast(TimestampType()))\
    .withColumn("DJJGeneratedFlag", F.lit(0).cast(IntegerType()))

# 4. TABLE LOAD (UPDATE / INSERT / DELETE)

In [0]:
%skip
# SQL_IdentityInsert - Execute SQL Task

# ============================================================
# CREATE UNKNOWN AND N/A RECORDS FOR DIMENSIONS 
# A "-1" value represents an unknown dimension value
# A "0" value represents a "Not Applicable" dimension value
# INSERT DEFAULT RECORDS
# ============================================================

Spark.sql(f"""
    INSERT INTO djj_enriched_non_prod.Brk.dim_Booking (
        BookingKey,
        BookingID,
        SalesOrderNo,
        PONo,
        BookingNumber,
        VesselName,
        Voyage,
        TotalContainers,
        Status,
        EstDepartureDate,
        EstArrivalDate,
        OBLNum,
        OBLDate,
        FundsReceiptDate,
        DJJGeneratedFlag,
        ETLSSExecutionID,
        DJJLastUpdateTime
    )
    VALUES
    (
        0,
        0,
        'NA',
        'NA',
        'Not Applicable',
        'Not Applicable',
        'Not Applicable',
        0,
        'Not Applicable',
        CAST('1900-01-01' AS DATE),
        CAST('1900-01-01' AS DATE),
        'Not Applicable',
        CAST('1900-01-01' AS DATE),
        CAST('1900-01-01' AS DATE),
        1,
        0,
        CURRENT_TIMESTAMP()
    ),
    (
        -1,
        -1,
        'Unknown',
        'Unknown',
        'Unknown',
        'Unknown',
        'Unknown',
        0,
        'Unknown',
        CAST('1900-01-01' AS DATE),
        CAST('1900-01-01' AS DATE),
        'Unknown',
        CAST('1900-01-01' AS DATE),
        CAST('1900-01-01' AS DATE),
        1,
        0,
        CURRENT_TIMESTAMP()
    );
""")

In [0]:
# ============================================================
# UPDATE AND INSERT RECORDS
# ============================================================

tf_up_dst_brk_dimbooking_df = der_metadata_output_df.select(
    F.col("BookingID").cast(IntegerType()).alias("BookingID"),
    F.substring(F.col("SalesOrderNo"), 1, 10).cast(StringType()).alias("SalesOrderNo"),
    F.substring(F.col("PONo"), 1, 10).cast(StringType()).alias("PONo"),
    F.substring(F.col("BookingNumber"), 1, 50).cast(StringType()).alias("BookingNumber"),
    F.substring(F.col("VesselName"), 1, 50).cast(StringType()).alias("VesselName"),
    F.substring(F.col("Voyage"), 1, 50).cast(StringType()).alias("Voyage"),
    F.col("TotalContainers").cast(IntegerType()).alias("TotalContainers"),
    F.substring(F.col("Status"), 1, 50).cast(StringType()).alias("Status"),
    F.col("EstDepartureDate").cast(TimestampType()).alias("EstDepartureDate"),
    F.col("EstArrivalDate").cast(TimestampType()).alias("EstArrivalDate"),
    F.substring(F.col("OBLNum"), 1, 50).cast(StringType()).alias("OBLNum"),
    F.col("OBLDate").cast(TimestampType()).alias("OBLDate"),
    F.col("FundsReceiptDate").cast(TimestampType()).alias("FundsReceiptDate"),
    F.col("DJJGeneratedFlag").cast(IntegerType()).alias("DJJGeneratedFlag"),
    F.col("ETLSSExecutionID").cast(IntegerType()).alias("ETLSSExecutionID"),
    F.col("DJJLastUpdateTime").cast(TimestampType()).alias("DJJLastUpdateTime")
)

tf_up_dst_brk_dimbooking_df.createOrReplaceTempView("source_view")

target_table_name = f"{DestinationDatabaseName}.{DestinationSchemaName}.{DestinationTableName}"

spark.sql(f"""
    MERGE INTO {target_table_name} AS target
    USING source_view AS source
    ON target.BookingID = source.BookingID
    AND target.SalesOrderNo = source.SalesOrderNo COLLATE UTF8_LCASE
    AND target.PONo = source.PONo COLLATE UTF8_LCASE
    AND target.BookingNumber = source.BookingNumber COLLATE UTF8_LCASE
    WHEN MATCHED AND (
        NOT(target.VesselName <=> source.VesselName COLLATE UTF8_LCASE) OR
        NOT(target.Voyage <=> source.Voyage COLLATE UTF8_LCASE) OR
        NOT(target.TotalContainers <=> source.TotalContainers) OR
        NOT(target.Status <=> source.Status COLLATE UTF8_LCASE) OR
        NOT(target.EstDepartureDate <=> source.EstDepartureDate) OR
        NOT(target.EstArrivalDate <=> source.EstArrivalDate) OR
        NOT(target.OBLNum <=> source.OBLNum COLLATE UTF8_LCASE) OR
        NOT(target.OBLDate <=> source.OBLDate) OR
        NOT(target.FundsReceiptDate <=> source.FundsReceiptDate) OR
        NOT(target.DJJGeneratedFlag <=> source.DJJGeneratedFlag)
    ) THEN UPDATE SET
        target.VesselName = source.VesselName,
        target.Voyage = source.Voyage,
        target.TotalContainers = source.TotalContainers,
        target.Status = source.Status,
        target.EstDepartureDate = source.EstDepartureDate,
        target.EstArrivalDate = source.EstArrivalDate,
        target.OBLNum = source.OBLNum,
        target.OBLDate = source.OBLDate,
        target.FundsReceiptDate = source.FundsReceiptDate,
        target.DJJGeneratedFlag = source.DJJGeneratedFlag,
        target.ETLSSExecutionID = source.ETLSSExecutionID,
        target.EnrichedTimestampUTC = source.DJJLastUpdateTime

    WHEN NOT MATCHED THEN INSERT (
        BookingID,
        SalesOrderNo,
        PONo,
        BookingNumber,
        VesselName,
        Voyage,
        TotalContainers,
        Status,
        EstDepartureDate,
        EstArrivalDate,
        OBLNum,
        OBLDate,
        FundsReceiptDate,
        DJJGeneratedFlag,
        ETLSSExecutionID,
        EnrichedTimestampUTC
    ) VALUES (
        source.BookingID,
        source.SalesOrderNo,
        source.PONo,
        source.BookingNumber,
        source.VesselName,
        source.Voyage,
        source.TotalContainers,
        source.Status,
        source.EstDepartureDate,
        source.EstArrivalDate,
        source.OBLNum,
        source.OBLDate,
        source.FundsReceiptDate,
        source.DJJGeneratedFlag,
        source.ETLSSExecutionID,
        source.DJJLastUpdateTime
    )
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

# 5. FINISH METADATA LOGGING

In [0]:
# ============================================================
# MERGE METRICS EXTRACTION
# ============================================================

merge_history = merge_history = spark.sql(f"""
    DESCRIBE HISTORY {DestinationDatabaseName}.{DestinationSchemaName}.{DestinationTableName}
""").filter("operation = 'MERGE'").first()
metrics = merge_history["operationMetrics"]
RowsInserted = int(metrics.get("numTargetRowsInserted", 0))
RowsUpdated = int(metrics.get("numTargetRowsUpdated", 0))
RowsDeleted = int(metrics.get("numTargetRowsDeleted", 0))

print(f"MERGE completed - Inserted: {RowsInserted}, Updated: {RowsUpdated}, Deleted: {RowsDeleted}")

print("ETL dim_booking completed successfully.")

In [0]:
# ==================================
# FINISH METADATA LOGGING
# ==================================

metrics = {
    "rows_inserted": RowsInserted,
    "rows_updated": RowsUpdated,
    "rows_deleted": RowsDeleted
}

end_execution_run(run_ctx, status="SUCCESS", metrics=metrics)